# drive2sim run $run_ids

Notebook do `d2s kaggle push` sinh ra. Chạy từ trên xuống; kết quả ghi vào `/kaggle/working/runs/` và `/kaggle/working/cache/`, tải về bằng `d2s kaggle pull <run_id>`.

In [ ]:
# Môi trường của phiên Kaggle: ghi lại để điền câu hỏi mở về phiên bản PyTorch.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch

print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
%%bash
# Lấy đúng commit đã tạo run. %%bash có set -e: lỗi thì notebook dừng và báo lỗi, không chạy tiếp
# các ô sau với code sai (lệnh "!" của IPython bỏ qua mã lỗi).
set -euo pipefail
git clone --quiet --recurse-submodules --branch $branch $repo_url /kaggle/tmp/repo
cd /kaggle/tmp/repo
git checkout --quiet $commit
git submodule update --quiet --init --recursive

In [ ]:
%%bash
# Cài môi trường chính bằng đúng script dùng trên máy cá nhân (envs/install_main.sh), để run trên
# Kaggle và trên máy dùng cùng phiên bản PyTorch, gsplat-geer, nvdiffrast, pytorch3d. Build cho
# đúng GPU của phiên này (compute capability đọc từ nvidia-smi, ví dụ 7.5 cho T4, 8.9 cho L4);
# mất khoảng vài chục phút mỗi phiên. Python của Kaggle (3.13 từ 11/2026) mới hơn bản requirements.txt
# hỗ trợ (torch 2.4.1, numpy 1.23.5), nên cài vào một venv Python 3.10 như envs/main.yml và Colab.
set -euo pipefail
cd /kaggle/tmp/repo
command -v uv >/dev/null || python -m pip install --quiet uv
uv venv --quiet --seed --python 3.10 /kaggle/tmp/venv
[ -d /usr/local/cuda/bin ] && export PATH="/usr/local/cuda/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | sort -u | paste -sd';')"
echo "TORCH_CUDA_ARCH_LIST=$TORCH_CUDA_ARCH_LIST"
# nvcc của image khác major với PyTorch thì install_main.sh cài CUDA 12.8 vào đây (~3 GB).
export D2S_CUDA_PREFIX=/kaggle/tmp/cuda-12.8
# Wheel đã build của gsplat-geer, nvdiffrast, pytorch3d... (mất khoảng 25 phút mỗi lần build): tìm
# trong output notebook trước đã gắn (/kaggle/input/**/wheels/<khoá>/), không có thì build rồi lưu vào
# output của notebook này cho lần sau (ARCHITECTURE mục 2.8).
export D2S_WHEEL_SOURCES=/kaggle/input D2S_WHEEL_CACHE=/kaggle/working/wheels
bash envs/install_main.sh --python /kaggle/tmp/venv/bin/python

In [ ]:
# Gói job (CONTRACTS mục 9.2): mỗi run một thư mục /kaggle/tmp/bundle/<run_id>/.
import json
import pathlib

bundles = json.loads(r"""$bundles_json""")
for run_id, bundle in bundles.items():
    bundle_dir = pathlib.Path("/kaggle/tmp/bundle") / run_id
    bundle_dir.mkdir(parents=True, exist_ok=True)
    for name, content in bundle.items():
        (bundle_dir / name).write_text(json.dumps(content, indent=2))

In [ ]:
%%bash
# Chạy run, rồi luôn dọn output kể cả khi run lỗi, để lần đẩy sau chạy tiếp được từ output này:
# output chỉ giữ runs/ và cache/, bỏ data/ (symlink tới dataset) và cảnh ảo bằng symlink.
# Mã lỗi của run được ô sau trả lại (sau khi in kết quả nhỏ vào log), nên notebook báo "error" khi run lỗi.
# Hai run (d2s kaggle push RUN1 RUN2): chuẩn bị thư mục làm việc lần lượt, rồi chạy song song, mỗi
# run một GPU; run sau cần cùng mask, SMPL thì chờ run trước ghi xong (D2S_CACHE_WAIT_S).
set -uo pipefail
cd /kaggle/tmp/repo
D2S=/kaggle/tmp/venv/bin/d2s
ARGS="--workdir /kaggle/working --input-root /kaggle/input"
bundles=(/kaggle/tmp/bundle/*/)
if [ ${#bundles[@]} -eq 1 ]; then
  $D2S kaggle run "${bundles[0]}" $ARGS
  rc=$?
else
  rc=0
  for b in "${bundles[@]}"; do $D2S kaggle run "$b" $ARGS --prepare-only || rc=1; done
  export D2S_CACHE_WAIT_S=43200
  pids=()
  i=0
  for b in "${bundles[@]}"; do
    name="$(basename "$b")"
    (CUDA_VISIBLE_DEVICES=$i $D2S kaggle run "$b" $ARGS 2>&1 | sed -u "s/^/[${name: -4}] /"; exit "${PIPESTATUS[0]}") &
    pids+=($!)
    i=$((i + 1))
  done
  for p in "${pids[@]}"; do wait "$p" || rc=1; done
fi
rm -rf /kaggle/working/data
find /kaggle/working/runs /kaggle/working/cache -type l -delete 2>/dev/null || true
# Mã lỗi để ô sau trả lại, sau khi in kết quả nhỏ vào log.
echo $rc > /kaggle/tmp/run_exit_code

In [ ]:
# In các file kết quả nhỏ của run (run.json, summary, metric, thống kê) vào log (stderr của tiến
# trình), để máy đọc bằng `kaggle kernels logs` (`d2s kaggle pull --light`) thay vì liệt kê hàng
# nghìn file của output (CONTRACTS mục 9.2). Rồi trả lại mã lỗi của run.
import base64
import json
import os
import pathlib

working = pathlib.Path("/kaggle/working")
light = {}
patterns = (
    "run.json", "env.json", "stages/*/done", "stages/*/summary.json", "stages/*/*metrics*.json",
    "stages/*/train_stats.json", "stages/*/profile_step*.json", "stages/*/densify_stats.jsonl",
)
for d2s in working.glob("runs/*/*/*/d2s"):
    for pattern in patterns:
        for path in d2s.glob(pattern):
            if path.is_file() and path.stat().st_size <= 4 << 20:
                light[path.relative_to(working).as_posix()] = path.read_text(errors="replace")
encoded = base64.b64encode(json.dumps(light).encode()).decode()
os.write(2, ("\nD2S_RUN_LIGHT " + encoded + "\n").encode())
code = int(pathlib.Path("/kaggle/tmp/run_exit_code").read_text())
if code:
    raise SystemExit(code)